# Distingue variabilidad de proceso e incertidumbre

Nivel: 1. Requisitos previos: Nivel 1 de aprendizaje. Tiempo de lectura: 18 minutos.

## Objetivos

Lee el registro de incertidumbre comprometido, identifica sus entradas inciertas y cuantiles de salida, y explica por qué un intervalo de un estado simulado no es un intervalo de confianza de planta.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Configura las importaciones y registra versiones. Este paso hace reproducible la ejecución del método.

In [1]:
# Importa herramientas de análisis y registra sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resuelve el pipeline local desde la estructura relativa del repositorio.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Datos

Responsable: autores de OreFlow; `data/derived/benchmark.json` y `data/derived/studies.json`, licencia MIT del proyecto. Los rangos son supuestos de incertidumbre aplicados a casos de proceso propios; las etiquetas son simuladas. Repositorio: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

OreFlow varía índice de trabajo, ley de cabeza, tamaño de liberación y flotabilidad dentro de rangos multiplicativos declarados, y resuelve nuevamente el mismo circuito estacionario. Un hipercubo latino con semilla genera muestras reproducibles. Los cuantiles describen este modelo y estos rangos; no estiman error de medición de terreno ni variabilidad entre plantas.

Carga los cuantiles de incertidumbre y el estudio de semillas repetidas para el mismo caso de autoría propia.

In [2]:
# Carga los cuantiles de incertidumbre y el estudio de semillas repetidas para el mismo caso de autoría propia.
repo = Path('../../../').resolve()
benchmark = json.loads((repo / 'data/derived/benchmark.json').read_text(encoding='utf-8'))
studies = json.loads((repo / 'data/derived/studies.json').read_text(encoding='utf-8'))
record = benchmark['uncertainty']['copper_porphyry_soft']
seed_study = studies['cases']['copper_porphyry_soft']['seed_study']
print('design:', seed_study['design'], '| samples:', seed_study['samples'], '| seeds:', len(seed_study['seeds']))
display(pd.DataFrame([{'output': name, **summary} for name, summary in record.items() if isinstance(summary, dict) and 'p50' in summary]).set_index('output'))

design: Latin hypercube | samples: 128 | seeds: 8


,p05,p50,p95
output,,,
recovery_pct,89.410940,91.897183,93.795676
concentrate_grade,24.868232,26.441278,27.709650


Compara la entrada que domina la dispersión de recuperación con el rango P05-P95. Son propiedades del dominio de simulación declarado.

In [3]:
# Compara la entrada que domina la dispersión de recuperación con el rango P05-P95.
print('dominant input by output:', record['dominant_input'])
print('all-constraint probability:', record['probabilities']['all_constraints'])
seed_rows = pd.DataFrame([{'seed': item['seed'], 'recovery P05': item['recovery_pct']['p05'], 'recovery P50': item['recovery_pct']['p50'], 'recovery P95': item['recovery_pct']['p95'], 'all constraints': item['all_constraints']} for item in seed_study['per_seed']])
display(seed_rows)

dominant input by output: {'recovery_pct': 'floatability', 'concentrate_grade': 'liberation_size', 'specific_energy_grinding_kwh_t': 'work_index', 'recovered_primary_tph': 'head_grade'}
all-constraint probability: 0.8203125


,seed,recovery P05,recovery P50,recovery P95,all constraints
0,101,89.216297,91.996411,94.025237,0.820312
1,202,89.018685,91.983648,94.034888,0.820312
2,303,89.042295,91.985040,93.914918,0.812500
3,404,89.367639,91.975672,93.875840,0.812500
4,505,89.079242,92.035955,93.745444,0.820312
5,606,89.087326,91.944077,93.799518,0.812500
6,707,89.374171,92.092473,94.007753,0.812500
7,808,89.294996,91.917077,93.948797,0.812500


## Resultados

Los cuantiles resumen la salida del modelo condicionada por las semiamplitudes de autoría y el caso elegido. El estudio de semillas muestra cómo cambian estimaciones con otro diseño. Ninguno describe incertidumbre observacional real sin mediciones independientes y un modelo de incertidumbre justificado.

## Resumen

Separaste un estudio de incertidumbre simulada con semilla de la incertidumbre de medición. Al citar resultados, declara diseño, rangos de entrada y procedencia de datos.

## Referencias

- https://doi.org/10.1080/00401706.1979.10489755